# D09 — train 5 model, 1 click

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/quocluongg/d09-eval-colab/blob/main/colab/run_all.ipynb)

**Cách chạy:** `Runtime` → `Run all` (Ctrl+F9). Không phải chỉnh sửa gì thêm.

Pipeline tự động: clone repo → tạo venv + cài đúng `package/requirements.lock.txt` → `selfcheck` → `run` (5 model, 21 lượt train) → `pack` → in báo cáo → tải `out.zip` về máy.

Lệnh nào lỗi thì pipeline **dừng ngay** và in nguyên văn lỗi, không tự sửa. Log nằm trong `logs/`.

Không sửa: `experiment.json`, `experiment.sha256`, `requirements.lock.txt`, `reference/`, `data/`, `package/src/`, `package/tests/`.

In [ ]:
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/quocluongg/d09-eval-colab.git"
REPO_DIR = pathlib.Path("/content/d09-eval-colab")

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
print("cwd    =", os.getcwd())
print("commit =", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("python =", sys.version.split()[0])

In [ ]:
import subprocess

proc = subprocess.Popen(
    ["bash", "colab/run_all.sh"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
if rc != 0:
    raise SystemExit(f"run_all.sh failed with exit code {rc} — dừng, không tự sửa. Đọc lỗi nguyên văn ở trên và trong logs/.")

In [ ]:
from pathlib import Path

print(Path("logs/RESULTS.txt").read_text(encoding="utf-8"))

from google.colab import files

files.download("out.zip")

## Gửi lại 4 thứ

1. `out.zip` do lệnh `pack` tạo — **không giải nén, không đổi tên**.
2. SHA-256 của zip: dòng `zip_sha256=` trong `logs/RESULTS.txt` (Windows: `Get-FileHash out.zip -Algorithm SHA256`).
3. Nguyên văn các dòng `selfcheck` — mục `=== SELFCHECK (verbatim) ===`.
4. Máy: `os=`, `ram_total_gb=`, và `run_wall_seconds=` — mục `=== MACHINE ===` / `=== DELIVERABLES ===`.

`out.zip` chứa 21 `predictions.csv` + 21 `run_meta.json` + `MANIFEST.json` (sha256 từng file).